# CBP CFP Core Feasibility Pilot — Real Faces

Recommended workflow:

- **GitHub** = source code, configs, tests, docs, notebook.
- **Google Drive** = raw CFP ZIP, embedding cache, experiment outputs.
- **Colab local disk** = cloned source + extracted CFP images for fast runtime I/O.

The notebook tests three first-stage CBP questions:

1. **Gate A / E1 — phenomenon:** do mean-similar states retain different evidence/influence on unseen identities?
2. **Gate B / E2 — decision relevance:** does evidence-aware CBP improve/preserve genuine yield versus matched mean-only compression at matched FMR/coverage?
3. **E3 — router fidelity:** does the inductive single-image router preserve teacher-policy behavior on unseen identities?

### Frozen FR model
Default: **InsightFace `buffalo_l`**, with the face-recognition backbone frozen.

> This is a **mechanism pilot** using CFP identities/captures, not the official CFP-FP pair-protocol benchmark score.

Before Gate A/B, the notebook now runs a **native InsightFace extraction audit** on frontal/profile CFP samples and blocks the full run if frontal detection is implausibly low.


In [ ]:
# ============================================================
# 0. USER CONFIG
# ============================================================

USE_GOOGLE_DRIVE = True

# Public source repository. No GitHub token is needed.
GITHUB_REPO_URL = "https://github.com/Tai12345-Ai/FIQA-CBP.git"
GITHUB_REF = "main"  # Commit + push local changes before opening Colab.

# Raw CFP ZIP remains on Google Drive.
CFP_ZIP_DRIVE = "/content/drive/MyDrive/datasets/CFP/cfp-dataset.zip"

# Fast local runtime locations.
PROJECT_DIR = "/content/FIQA-CBP"
CFP_LOCAL_DIR = "/content/CFP"

DATASET_NAME = "CFP_core_pilot"
MODEL_NAME = "buffalo_l"

# Fresh cache names for the validated CFP protocol:
# +25% BORDER_REPLICATE context padding before native InsightFace detection.
EMBEDDINGS_NPZ = "/content/drive/MyDrive/CBP/cache/cfp_buffalo_l_pad25_native_v1.npz"
PILOT_EMBEDDINGS_NPZ = "/content/drive/MyDrive/CBP/cache/cfp_buffalo_l_pad25_native_pilot.npz"
RUN_OUTPUT_DIR = "/content/drive/MyDrive/CBP/runs/cfp_buffalo_l_core_pilot"

# Detection protocol validated by the CFP pilot diagnostic.
PILOT_N_PER_VIEW = 50
PILOT_MIN_FRONTAL_DET_RATE = 0.90  # engineering sanity gate, not a paper metric
DET_SIZE = (640, 640)
DET_THRESH = 0.5
PADDING_RATIO = 0.25
REQUIRE_CUDA = True

# True = smoke test only; do not use as research evidence.
FAST_MODE = False

# Run pytest + E0 before the real-face pilot.
RUN_CODE_SANITY = True

# True only when intentionally regenerating embeddings.
FORCE_REEXTRACT = False

SEED = 20261021


In [ ]:
# ============================================================
# 1. Mount Drive + clone source from GitHub to local Colab disk
# ============================================================

import os, sys, shutil, subprocess, json, time
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    shutil.rmtree(project_dir)

# Public repository: no credentials or Colab Secret required.
subprocess.check_call([
    "git", "clone", "--depth", "1", "--branch", GITHUB_REF,
    GITHUB_REPO_URL, str(project_dir),
])

commit_sha = subprocess.check_output(
    ["git", "-C", str(project_dir), "rev-parse", "HEAD"],
    text=True,
).strip()

assert (project_dir / "pyproject.toml").exists(), "Cloned repo is missing pyproject.toml"

PROJECT_DIR = project_dir
os.chdir(PROJECT_DIR)

print("PROJECT_DIR =", PROJECT_DIR)
print("Git ref     =", GITHUB_REF)
print("Commit SHA  =", commit_sha)
print("Python      =", sys.version)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true"],
    capture_output=True, text=True
).stdout.strip()
print("GPU         =", gpu if gpu else "No GPU detected")


In [ ]:
# ============================================================
# 2. Install dependencies + verify NVIDIA/ONNX Runtime GPU
# ============================================================

import subprocess, sys, importlib.metadata

# Project dependencies. The project's [dev] extra does not install ONNX Runtime.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"
])

# InsightFace/OpenCV.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "insightface==0.7.3", "opencv-python-headless"
])

# Colab currently exposes CUDA 13 on L4. Remove conflicting CPU/GPU ORT
# distributions, then install a CUDA-13-compatible GPU build.
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "onnxruntime", "onnxruntime-gpu"
], check=False, stdout=subprocess.DEVNULL)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "onnxruntime-gpu>=1.27"
])

# Import only after the final ORT package has been installed.
import torch
import onnxruntime as ort

print("torch version       :", torch.__version__)
print("torch CUDA build    :", torch.version.cuda)
print("torch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                 :", torch.cuda.get_device_name(0))

print("onnxruntime version :", ort.__version__)
print("ORT providers       :", ort.get_available_providers())

try:
    print("onnxruntime CPU pkg :", importlib.metadata.version("onnxruntime"))
except importlib.metadata.PackageNotFoundError:
    print("onnxruntime CPU pkg : not installed")
print("onnxruntime-gpu pkg :", importlib.metadata.version("onnxruntime-gpu"))

assert torch.cuda.is_available(), (
    "PyTorch cannot see an NVIDIA GPU. Select an NVIDIA GPU runtime in Colab."
)

PROVIDER = (
    "CUDAExecutionProvider"
    if "CUDAExecutionProvider" in ort.get_available_providers()
    else "CPUExecutionProvider"
)
print("InsightFace provider:", PROVIDER)

if REQUIRE_CUDA:
    assert PROVIDER == "CUDAExecutionProvider", (
        "NVIDIA GPU is present, but ONNX Runtime did not expose CUDAExecutionProvider. "
        "If ONNX Runtime had already been imported earlier in this session, restart "
        "the Colab session once and rerun from Cell 0."
    )


## 3. CFP dataset preflight

Bạn chỉ cần upload **nguyên file ZIP** từ Kaggle lên Google Drive.

Notebook sẽ:

1. kiểm tra file `CFP_ZIP_DRIVE`;
2. giải nén ZIP vào `/content/CFP` trên Colab;
3. tự tìm `cfp-dataset/Data/Images`;
4. build identity manifest;
5. kiểm tra số identities/captures trước khi chạy InsightFace.

Cách này nhanh hơn nhiều so với upload 7,000 file ảnh riêng lẻ lên Drive.


In [ ]:
# ============================================================
# 3. Unzip CFP locally + strict structure check + manifest
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import subprocess, sys, os, json, shutil, zipfile

zip_path = Path(CFP_ZIP_DRIVE)
assert zip_path.exists(), f"Không thấy CFP ZIP trên Drive: {zip_path}"

local_root = Path(CFP_LOCAL_DIR)

# Giải nén vào local Colab disk để đọc nhanh hơn Google Drive.
if not local_root.exists() or not any(local_root.iterdir()):
    local_root.mkdir(parents=True, exist_ok=True)
    print("Extracting CFP ZIP to local runtime...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_root)
else:
    print("Reuse existing local CFP extraction:", local_root)

candidates = [
    local_root / "cfp-dataset" / "Data" / "Images",
    local_root / "Data" / "Images",
]
candidates += list(local_root.glob("*/cfp-dataset/Data/Images"))
candidates += list(local_root.glob("*/Data/Images"))

image_root = next((p for p in candidates if p.exists()), None)
if image_root is None:
    print("Không tìm thấy Data/Images. Các mục cấp 1 sau khi giải nén:")
    for p in sorted(local_root.iterdir()):
        print(" -", p.name)
    raise FileNotFoundError(
        "Không tìm thấy cấu trúc CFP raw */Data/Images sau khi giải nén ZIP."
    )

cfp_root = image_root.parent.parent

print("ZIP           :", zip_path)
print("Local root    :", local_root)
print("CFP root      :", cfp_root)
print("Image root    :", image_root)
print("Protocol found:", (cfp_root / "Protocol").exists())
print("Readme found  :", (cfp_root / "Readme.txt").exists())

identity_dirs = sorted([p for p in image_root.iterdir() if p.is_dir()])
print("Identity folders detected:", len(identity_dirs))
print("First identities:", [p.name for p in identity_dirs[:10]])
assert len(identity_dirs) > 0, f"Không thấy identity folders trong {image_root}"

data_dir = PROJECT_DIR / "data"
data_dir.mkdir(exist_ok=True)
manifest_path = data_dir / f"{DATASET_NAME}_manifest.csv"

from cbp.data.manifest import FaceManifest

manifest = FaceManifest.from_identity_folders(image_root)
manifest.to_csv(manifest_path)
print(f"Manifest written: {len(manifest.frame)} rows -> {manifest_path}")

df = manifest.frame.copy()
required = {"image_path", "identity_id", "sample_id"}
assert required.issubset(df.columns), f"Manifest thiếu cột: {required - set(df.columns)}"

counts = df.groupby("identity_id").size().sort_values()
sample_id_lower = df["sample_id"].astype(str).str.lower()
is_frontal = sample_id_lower.str.contains(":frontal/", regex=False)
is_profile = sample_id_lower.str.contains(":profile/", regex=False)

summary = {
    "images": int(len(df)),
    "identities": int(df["identity_id"].nunique()),
    "frontal": int(is_frontal.sum()),
    "profile": int(is_profile.sum()),
    "identities_ge_2_captures": int((counts >= 2).sum()),
    "median_captures_per_identity": float(counts.median()),
    "min_captures": int(counts.min()),
    "max_captures": int(counts.max()),
}
print(json.dumps(summary, indent=2))

# Strict CFP sanity gate for the Kaggle copy used by this project.
assert summary["images"] == 7000, f"Expected 7000 CFP images, got {summary['images']}"
assert summary["identities"] == 500, f"Expected 500 identities, got {summary['identities']}"
assert summary["frontal"] == 5000, f"Expected 5000 frontal images, got {summary['frontal']}"
assert summary["profile"] == 2000, f"Expected 2000 profile images, got {summary['profile']}"

print("\nCFP sanity PASS: 7000 images = 5000 frontal + 2000 profile, 500 identities.")
display(counts.describe().to_frame("captures_per_identity"))


## 4. Native InsightFace preprocessing audit

CFP provides very tight face crops. A diagnostic sweep showed that native SCRFD detection at the standard threshold is poor without context, while **+25% replicate-border padding** restores high detection coverage without lowering the standard threshold.

Run 1 therefore uses one fixed protocol for every CFP sample:

```
CFP crop
-> +25% BORDER_REPLICATE padding on each side
-> InsightFace FaceAnalysis detection
-> native 5-point alignment
-> native recognition preprocessing (112x112)
-> buffalo_l embedding
```

Before any CBP experiment, this section:

1. samples 50 frontal + 50 profile images;
2. runs the fixed padded protocol;
3. reports detection coverage separately for frontal/profile;
4. visualizes remaining failures;
5. only then permits the full 7,000-image extraction.


In [ ]:
# ============================================================
# 4A. 100-image native InsightFace pilot audit
# ============================================================

import pandas as pd
import subprocess, sys, json
from pathlib import Path
import onnxruntime as ort

AVAILABLE_PROVIDERS = ort.get_available_providers()
print("ORT providers:", AVAILABLE_PROVIDERS)
assert "CUDAExecutionProvider" in AVAILABLE_PROVIDERS, (
    "CUDAExecutionProvider chưa hoạt động. Không chạy pilot bằng CPU."
)
PROVIDER = "CUDAExecutionProvider"
print("Using provider:", PROVIDER)

frontal_df = df[is_frontal].sample(
    n=min(PILOT_N_PER_VIEW, int(is_frontal.sum())),
    random_state=SEED,
)
profile_df = df[is_profile].sample(
    n=min(PILOT_N_PER_VIEW, int(is_profile.sum())),
    random_state=SEED + 1,
)
pilot_df = pd.concat([frontal_df, profile_df], ignore_index=True)

pilot_manifest_path = data_dir / "cfp_native_insightface_detection_pilot.csv"
pilot_df.to_csv(pilot_manifest_path, index=False)

pilot_emb_path = Path(PILOT_EMBEDDINGS_NPZ)
pilot_emb_path.parent.mkdir(parents=True, exist_ok=True)
pilot_meta_path = Path(str(pilot_emb_path) + ".metadata.json")
pilot_failure_path = Path(str(pilot_emb_path) + ".failures.tsv")

print("Pilot manifest:", pilot_manifest_path)
print("Pilot samples :", len(pilot_df))
print("Frontal       :", int(pilot_df["sample_id"].str.contains(":frontal/", case=False, regex=False).sum()))
print("Profile       :", int(pilot_df["sample_id"].str.contains(":profile/", case=False, regex=False).sum()))
print("Padding ratio :", PADDING_RATIO)
print("Detection thr :", DET_THRESH)

cmd = [
    sys.executable, "-m", "scripts.extract_embeddings",
    "--manifest", str(pilot_manifest_path),
    "--output", str(pilot_emb_path),
    "--model", MODEL_NAME,
    "--provider", PROVIDER,
    "--ctx-id", "0",
    "--det-size", str(DET_SIZE[0]), str(DET_SIZE[1]),
    "--det-thresh", str(DET_THRESH),
    "--padding-ratio", str(PADDING_RATIO),
    "--metadata", str(pilot_meta_path),
    "--failure-log", str(pilot_failure_path),
]

print("\nRunning padded native InsightFace pilot audit...")
subprocess.check_call(cmd, cwd=PROJECT_DIR)

pilot_meta = json.loads(pilot_meta_path.read_text(encoding="utf-8"))
print("\nPipeline:", pilot_meta.get("pipeline"))
print("Recognition input size:", pilot_meta.get("recognition_input_size"))
print("Pre-detector resize:", pilot_meta.get("pre_detector_resize"))
print("Pre-detector normalization:", pilot_meta.get("pre_detector_normalization"))
print("Pre-detector padding ratio:", pilot_meta.get("pre_detector_padding_ratio"))
print("Pre-detector padding mode :", pilot_meta.get("pre_detector_padding_mode"))

view_audit = pilot_meta.get("view_audit", {})
audit_table = pd.DataFrame(view_audit).T
display(audit_table)

frontal_rate = view_audit.get("frontal", {}).get("detection_rate")
profile_rate = view_audit.get("profile", {}).get("detection_rate")

print("Frontal detection rate:", frontal_rate)
print("Profile detection rate:", profile_rate)

if frontal_rate is None or frontal_rate < PILOT_MIN_FRONTAL_DET_RATE:
    raise RuntimeError(
        f"STOP: frontal detection rate {frontal_rate} is below the engineering "
        f"sanity gate {PILOT_MIN_FRONTAL_DET_RATE:.0%}. Do not run full CFP yet."
    )

print("\nPilot frontal sanity PASS. Full CFP extraction may proceed.")


In [ ]:
# ============================================================
# 4B. Inspect pilot failures visually
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

if pilot_failure_path.exists() and pilot_failure_path.stat().st_size > 0:
    pilot_fail = pd.read_csv(pilot_failure_path, sep="\t")
else:
    pilot_fail = pd.DataFrame()

if pilot_fail.empty:
    print("No pilot failures.")
else:
    print("Failure counts:")
    display(
        pilot_fail.groupby(["view", "failure_type"])
        .size()
        .rename("count")
        .to_frame()
    )

    show_fail = pilot_fail.head(12).reset_index(drop=True)
    cols = 4
    rows = (len(show_fail) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for i, row in show_fail.iterrows():
        image = Image.open(row["image_path"]).convert("RGB")
        axes[i].imshow(image)
        axes[i].set_title(f"{row['view']} | {row['failure_type']}", fontsize=9)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 4C. Full 7,000-image padded native InsightFace extraction
# ============================================================

from pathlib import Path
import subprocess, sys, json
import pandas as pd
import onnxruntime as ort

AVAILABLE_PROVIDERS = ort.get_available_providers()
assert "CUDAExecutionProvider" in AVAILABLE_PROVIDERS, (
    "CUDAExecutionProvider chưa hoạt động. Không chạy full extraction bằng CPU."
)
PROVIDER = "CUDAExecutionProvider"

emb_path = Path(EMBEDDINGS_NPZ)
emb_path.parent.mkdir(parents=True, exist_ok=True)

if emb_path.exists() and not FORCE_REEXTRACT:
    print("Reuse padded native InsightFace embedding cache:", emb_path)
else:
    cmd = [
        sys.executable, "-m", "scripts.extract_embeddings",
        "--manifest", str(manifest_path),
        "--output", str(emb_path),
        "--model", MODEL_NAME,
        "--provider", PROVIDER,
        "--ctx-id", "0",
        "--det-size", str(DET_SIZE[0]), str(DET_SIZE[1]),
        "--det-thresh", str(DET_THRESH),
        "--padding-ratio", str(PADDING_RATIO),
        "--expected-identities", "500",
        "--metadata", str(emb_path) + ".metadata.json",
        "--failure-log", str(emb_path) + ".failures.tsv",
    ]
    print("Running full padded native InsightFace extraction...")
    subprocess.check_call(cmd, cwd=PROJECT_DIR)

print("Embeddings:", emb_path)
print("Size MB:", round(emb_path.stat().st_size / 1024**2, 2))

meta_path = Path(str(emb_path) + ".metadata.json")
assert meta_path.exists(), meta_path
meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("\n=== FULL EXTRACTION SUMMARY ===")
print("Pipeline            :", meta.get("pipeline"))
print("Padding ratio       :", meta.get("pre_detector_padding_ratio"))
print("Padding mode        :", meta.get("pre_detector_padding_mode"))
print("Detection threshold :", meta.get("det_thresh"))
print("Samples expected    :", meta.get("samples_expected"))
print("Samples succeeded   :", meta.get("samples_succeeded"))
print("Samples failed      :", meta.get("samples_failed"))
print("Success fraction    :", meta.get("success_fraction"))
print("Identities succeeded:", meta.get("identities_succeeded"))

full_view_audit = meta.get("view_audit", {})
display(pd.DataFrame(full_view_audit).T)

full_failure_path = Path(str(emb_path) + ".failures.tsv")
if full_failure_path.exists():
    full_fail = pd.read_csv(full_failure_path, sep="\t")
    if not full_fail.empty:
        print("\nFull failure counts:")
        display(
            full_fail.groupby(["view", "failure_type"])
            .size()
            .rename("count")
            .to_frame()
        )


In [ ]:
# ============================================================
# 5. Optional code/math sanity
# ============================================================

import subprocess, sys, json
from pathlib import Path

if RUN_CODE_SANITY:
    print("pytest...")
    subprocess.check_call([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR)

    e0_out = PROJECT_DIR / "outputs" / "e0_colab" / "report.json"
    e0_out.parent.mkdir(parents=True, exist_ok=True)

    print("\nE0 exact-vs-Monte-Carlo...")
    subprocess.check_call([
        sys.executable, "-m", "scripts.run_e0_synthetic",
        "--states", "16",
        "--identities", "50",
        "--repetitions", "5000" if FAST_MODE else "20000",
        "--seed", str(SEED),
        "--output", str(e0_out),
    ], cwd=PROJECT_DIR)

    e0 = json.loads(e0_out.read_text(encoding="utf-8"))
    print("E0 relative variance error:", e0.get("relative_error"))
else:
    print("Skipped code/math sanity.")


## 6. Focused CBP pilot config

Để trả lời **core feasibility** trước, notebook giảm trọng số của phần paper-scale:

- vẫn giữ identity-disjoint split;
- vẫn giữ CBP vs mean-only / basic controls;
- vẫn train router và refit deployed moments;
- ưu tiên FMR \(10^{-2}\), giữ \(10^{-3}\) như secondary;
- continuous/full-image neural comparator được tắt trong core pilot để giảm compute.

Sau khi Gate A/B có signal mới bật full Gate C/D.

In [ ]:
# ============================================================
# 6. Create focused pilot config
# ============================================================

import yaml
from pathlib import Path

base_cfg = yaml.safe_load((PROJECT_DIR / "configs" / "pilot.yaml").read_text(encoding="utf-8"))

base_cfg["seed"] = SEED
base_cfg["embedding_npz"] = str(emb_path)
base_cfg["output_dir"] = RUN_OUTPUT_DIR

# Core feasibility: không cần full expensive challenge ngay.
base_cfg["run_continuous_baseline"] = False
base_cfg["run_image_neural_baseline"] = False
base_cfg["certify_candidate_family"] = False

# Primary core pilot
base_cfg["k_candidates"] = [4, 8, 16]
base_cfg["n_states"] = 64
base_cfg["spectral_dim"] = 8

if FAST_MODE:
    base_cfg["max_identities_per_split"] = min(base_cfg.get("max_identities_per_split") or 80, 80)
    base_cfg["router_epochs"] = 20
    base_cfg["router_patience"] = 5
    base_cfg["phenomenon_permutations"] = 100
    base_cfg["bootstrap_repetitions"] = 100
    print("FAST_MODE=True: smoke test only.")
else:
    base_cfg["phenomenon_permutations"] = 1000
    base_cfg["bootstrap_repetitions"] = 1000

pilot_cfg = PROJECT_DIR / "configs" / "colab_core_pilot.yaml"
pilot_cfg.write_text(yaml.safe_dump(base_cfg, sort_keys=False), encoding="utf-8")
print(pilot_cfg.read_text(encoding="utf-8"))


In [ ]:
# ============================================================
# 7. Run REAL-FACE core pilot
# ============================================================

import time
from pathlib import Path

from cbp.config import ExperimentConfig
from cbp.experiment import run_experiment
from cbp.fr.embeddings import EmbeddingTable

# Preflight: fail early with a useful traceback/message instead of a generic
# subprocess CalledProcessError.
pilot_cfg = Path(pilot_cfg)
assert pilot_cfg.exists(), f"Missing pilot config: {pilot_cfg}"
assert Path(emb_path).exists(), f"Missing embedding cache: {emb_path}"

resolved_cfg = ExperimentConfig.from_yaml(pilot_cfg)
table_check = EmbeddingTable.load(resolved_cfg.embedding_npz)

print("Config      :", pilot_cfg)
print("Embeddings  :", resolved_cfg.embedding_npz)
print("Samples     :", len(table_check.embeddings))
print("Identities  :", len(set(table_check.identity_ids.tolist())))
print("Embedding D :", table_check.embeddings.shape[1])
print("Output      :", resolved_cfg.output_dir)

start = time.time()
report = run_experiment(pilot_cfg)

print(f"Finished in {(time.time()-start)/60:.1f} min")
print("Selected:", report["selection"]["selected"])
print("Gates:", report["gates"])
print("Output:", RUN_OUTPUT_DIR)


In [ ]:
# ============================================================
# 8. Read only the core scientific evidence first
# ============================================================

from pathlib import Path
import json

report_path = Path(RUN_OUTPUT_DIR) / "report.json"
assert report_path.exists(), report_path
report = json.loads(report_path.read_text(encoding="utf-8"))

gates = report.get("gates", {})

print("=== GATE A: REAL-FACE PHENOMENON ===")
print(json.dumps(gates.get("A_phenomenon", {}), indent=2)[:15000])

print("\n=== GATE B: DECISION RELEVANCE ===")
print(json.dumps(gates.get("B_decision_relevance", {}), indent=2)[:15000])

print("\n=== SELECTED REPRESENTATION ===")
print(json.dumps(report.get("selection", {}), indent=2)[:10000])

# Tìm mọi router_transfer block trong report mà không phụ thuộc exact nesting.
router_blocks = []
def walk(x, path="root"):
    if isinstance(x, dict):
        for k, v in x.items():
            p = f"{path}.{k}"
            if k == "router_transfer":
                router_blocks.append((p, v))
            walk(v, p)
    elif isinstance(x, list):
        for i, v in enumerate(x):
            walk(v, f"{path}[{i}]")

walk(report)

print("\n=== ROUTER TRANSFER BLOCKS ===")
for path, block in router_blocks[:20]:
    print("\n", path)
    print(json.dumps(block, indent=2)[:5000])


In [ ]:
# ============================================================
# 9. Core feasibility verdict
# ============================================================

def passed(name):
    x = gates.get(name, {})
    return x.get("pass") if isinstance(x, dict) else None

A = passed("A_phenomenon")
B = passed("B_decision_relevance")

print("CBP CORE REAL-FACE PILOT")
print("=" * 50)
print("Gate A phenomenon       :", "PASS" if A else "FAIL/NOT ESTABLISHED")
print("Gate B decision relevance:", "PASS" if B else "FAIL/NOT ESTABLISHED")

if A is True and B is True:
    print("\nCORE FEASIBILITY SIGNAL: POSITIVE")
    print("Next: second frozen backbone + full dense/continuous challenge + larger dataset.")
elif A is True and B is not True:
    print("\nPhenomenon có signal nhưng chưa có downstream decision value.")
    print("Không nên tăng compute để ép thắng; inspect representation/policy.")
else:
    print("\nCore phenomenon chưa được establish trên pilot này.")
    print("Inspect dataset, state construction, controls và backbone trước khi mở rộng.")

print("\nGate C/D nếu có trong report chỉ xem exploratory ở notebook này.")


In [ ]:
# ============================================================
# 10. Export run bundle
# ============================================================

import shutil
from pathlib import Path

run_dir = Path(RUN_OUTPUT_DIR)
zip_base = run_dir.parent / (run_dir.name + "_export")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=run_dir)
print("Exported:", zip_path)
